Install Required Libraries

In [11]:
!pip install -q transformers datasets peft accelerate bitsandbytes
!pip install -U bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 30.1 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0


Import Libraries

In [1]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model

Create a Simple Dataset

In [2]:
data = {
    "text": [
        "Question: What is machine learning? Answer: Machine learning is a field of AI that allows systems to learn from data.",
        "Question: What is deep learning? Answer: Deep learning is a subset of machine learning using neural networks.",
        "Question: What is overfitting? Answer: Overfitting happens when a model memorizes training data instead of learning patterns.",
        "Question: What is RAG? Answer: Retrieval Augmented Generation combines LLMs with external knowledge retrieval."
    ]
}

dataset = Dataset.from_dict(data)

Load Lightweight Model

In [3]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Tokenize Dataset


In [6]:
def tokenize(example):
    result = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    result["labels"] = result["input_ids"]
    return result

dataset = dataset.map(tokenize)
dataset.set_format(type="torch")
dataset = dataset.map(tokenize)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Configure LoRA

In [7]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model = get_peft_model(model, lora_config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:78: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from 'TinyLlama/TinyLlama-1.1B-Chat-v1.0' to 'None'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Training Setup

In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    num_train_epochs=1,
    logging_steps=1,
    learning_rate=2e-4,
    fp16=True,
    report_to="none"
)

Trainer

In [11]:
trainer = Trainer(
    model=model,
    train_dataset=dataset,
    args=training_args
)

trainer.train()

Step,Training Loss
1,4.772839
2,5.611070
3,5.405458
4,4.917408


TrainOutput(global_step=4, training_loss=5.176693797111511, metrics={'train_runtime': 3.8151, 'train_samples_per_second': 1.048, 'train_steps_per_second': 1.048, 'total_flos': 6362964688896.0, 'train_loss': 5.176693797111511, 'epoch': 1.0})

Test the Model

In [12]:
prompt = "Question: What is machine learning? Answer:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(**inputs, max_new_tokens=50)

print(tokenizer.decode(output[0]))

<s> Question: What is machine learning? Answer: Machine learning is a field of computer science that involves the use of algorithms to learn from data without being explicitly programmed. It is a powerful tool for analyzing large datasets and making predictions based on the patterns and relationships in the data. Machine learning algorithms


Save Model

In [13]:
model.save_pretrained("fine_tuned_model")
tokenizer.save_pretrained("fine_tuned_model")

('fine_tuned_model/tokenizer_config.json',
 'fine_tuned_model/chat_template.jinja',
 'fine_tuned_model/tokenizer.json')

Load the Fine-Tuned Model

In [16]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = "fine_tuned_model"

tokenizer = AutoTokenizer.from_pretrained(base_model)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    device_map="auto"
)

model = PeftModel.from_pretrained(model, adapter_path)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:598: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.2.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.2.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.2.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.2.self_attn.v_proj.l

Test the Model


In [17]:
prompt = "Question: What is machine learning? Answer:"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(
    **inputs,
    max_new_tokens=60
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Question: What is machine learning? Answer: Machine learning is a field of computer science that involves the use of algorithms to learn from data without being explicitly programmed. It is a powerful tool for analyzing large datasets and making predictions based on the patterns and relationships in the data. Machine learning algorithms can be used to identify patterns in data, make


Create an Interactive Chat Loop


In [21]:
while True:
    prompt = input("Ask a question: ")

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=80
    )

    print("\nAnswer:")
    print(tokenizer.decode(output[0], skip_special_tokens=True))

Ask a question: Question: What is machine learning? Answer:

Answer:
Question: What is machine learning? Answer: Machine learning is a field of computer science that involves the use of algorithms to learn from data without being explicitly programmed. It is a powerful tool for analyzing large datasets and making predictions based on the patterns and relationships in the data. Machine learning algorithms can be used to identify patterns in data, make predictions, and make decisions based on the results.
Ask a question: Question: What is deep learning? Answer:

Answer:
Question: What is deep learning? Answer: Deep learning is a type of machine learning that involves learning from large amounts of data. It is a form of artificial intelligence that allows machines to learn and improve their skills over time. Deep learning is often used in areas such as natural language processing, image recognition, and speech recognition.
Ask a question: Question: What is overfitting in machine learning?

KeyboardInterrupt: Interrupted by user

Download the Model From Colab

In [22]:
from google.colab import files
import shutil

shutil.make_archive("fine_tuned_model", 'zip', "fine_tuned_model")
files.download("fine_tuned_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>